# PSM v2: Morning vs Evening Response Time — Hard Reviews at Review Time

## What changed from v1

**v1 problem:** "Hard cards" were defined using `cards.factor` — the ease factor *today* (April 2026). Cards that are currently hard had most of their reviews in early 2025 when you were just starting out, creating a learning-curve confound.

**v2 fix:** A review is "hard" if `revlog.factor < 2.5` — i.e., Anki had already downgraded the card's ease factor **below the 2.5 default at the moment of that review**. This means the card had been failed or rated Hard enough times that Anki itself classified it as difficult. The definition is now temporally correct.

**Why ef < 2.5 (not a percentile)?** The ease factor distribution spikes at exactly 2.5 (the default starting value, 30.9% of reviews). Splitting that spike with a percentile cut is arbitrary. The natural boundary is below 2.5 = Anki has degraded the card; at or above 2.5 = still at default or improving.

| | v1 | v2 |
|---|---|---|
| Hard-card definition | cards.factor ≤ 2.65 (current snapshot) | revlog.factor < 2.5 (at review time) |
| Hard reviews (morning+evening) | 7,592 | 4,252 |
| Morning hard reviews | 545 | 319 |
| % of reviews from 2025 | ~82% | ~72% |

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.labelsize': 11,
    'axes.titlesize': 12,
})
MORNING_COLOR = '#2196F3'
EVENING_COLOR = '#FF9800'

DB_PATH   = r'C:\Users\sid99\AppData\Roaming\Anki2\Pottapatri\collection.anki2'
ANKI_CAP  = 60_000
HARD_THRESH = 2.5  # reviews with ef strictly below this are 'hard at review time'

In [ ]:
# ── Load and filter ────────────────────────────────────────────────────────
con = sqlite3.connect(DB_PATH)
con.create_collation('unicase', lambda a, b: (a > b) - (a < b))
revlog = pd.read_sql(
    'SELECT id, cid, ease, ivl, lastIvl, factor, time, type FROM revlog', con
)
con.close()

revlog['ts']    = pd.to_datetime(revlog['id'], unit='ms')
revlog['hour']  = revlog['ts'].dt.hour
revlog['month'] = revlog['ts'].dt.to_period('M')
revlog['ef']    = revlog['factor'] / 1000

# Hard reviews: ease factor already degraded below default AT review time
# ef == 0 means the card was still in the initial learning phase — exclude
df = revlog[
    (revlog['ef'] > 0) &
    (revlog['ef'] < HARD_THRESH) &
    (revlog['hour'].between(6, 11) | revlog['hour'].between(18, 23))
].copy()

df['morning'] = df['hour'].between(6, 11).astype(int)
df['capped']  = (df['time'] == ANKI_CAP).astype(int)

print(f'Hard reviews (ef < {HARD_THRESH}), morning + evening: {len(df):,}')
print(f'  Morning (06-12): {df["morning"].sum():,}')
print(f'  Evening (18-24): {(df["morning"]==0).sum():,}')
print(f'  Capped at 60s:   {df["capped"].sum():,} ({df["capped"].mean():.1%})')

In [ ]:
# ── Temporal distribution check ────────────────────────────────────────────
# Key question: are morning and evening hard reviews spread across the same
# time periods, or is one group concentrated in early (slower) months?
monthly = (
    df.groupby(['month', 'morning'])
      .size()
      .unstack(fill_value=0)
      .rename(columns={0: 'Evening', 1: 'Morning'})
      .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

months = [str(m) for m in monthly['month']]
x = np.arange(len(months))

axes[0].bar(x, monthly['Evening'], color=EVENING_COLOR, alpha=0.8, label='Evening')
axes[0].bar(x, monthly['Morning'], bottom=monthly['Evening'],
            color=MORNING_COLOR, alpha=0.8, label='Morning')
axes[0].set_ylabel('Hard reviews')
axes[0].set_title('Monthly Hard Reviews (ef < 2.5) by Time of Day')
axes[0].legend(frameon=False)

# Morning share
total = monthly['Morning'] + monthly['Evening']
morning_share = monthly['Morning'] / total.replace(0, np.nan)
axes[1].bar(x, morning_share, color=MORNING_COLOR, alpha=0.75)
axes[1].axhline(df['morning'].mean(), ls='--', color='grey', lw=1,
                label=f'Overall morning share ({df["morning"].mean():.1%})')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].set_ylabel('Morning share')
axes[1].set_xlabel('Month')
axes[1].legend(frameon=False)

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(months, rotation=40, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('v2_fig1_temporal.png', bbox_inches='tight')
plt.show()

print('Note: Zero morning hard reviews in Jan-Apr 2025 and Oct 2025.')
print('Morning hard-card reviewing only started May 2025.')

In [ ]:
# ── Feature engineering ────────────────────────────────────────────────────
df['abs_ivl'] = df['ivl'].abs()
df['log_ivl'] = np.log1p(df['abs_ivl'])

df = df.sort_values(['cid', 'ts'])
df['prev_ts']       = df.groupby('cid')['ts'].shift(1)
df['days_since']    = (df['ts'] - df['prev_ts']).dt.total_seconds() / 86400
df['log_days_since']= np.log1p(df['days_since'].fillna(df['days_since'].median()))

# Month index as numeric — lets us partially control for the learning curve
min_month = df['ts'].dt.to_period('M').min()
df['month_idx'] = (
    (df['ts'].dt.year - df['ts'].dt.year.min()) * 12 +
    df['ts'].dt.month - df['ts'].dt.month.min()
)

COVARIATES = ['ef', 'log_ivl', 'log_days_since', 'month_idx']

# Primary: exclude capped reviews
df_trim = df[df['capped'] == 0].dropna(subset=COVARIATES + ['time', 'morning']).copy()

print(f'After excluding capped: {len(df_trim):,}')
print(f'  Morning: {df_trim["morning"].sum():,}')
print(f'  Evening: {(df_trim["morning"]==0).sum():,}')
print()
print('Response time (capped excluded):')
print(df_trim.groupby('morning')['time']
        .agg(['median', 'mean', 'count'])
        .rename(index={0: 'Evening', 1: 'Morning'})
        .round(0))

---
## Propensity Score Estimation

Confounders: ease factor at review time (`ef`), log-interval, log-days since last review, and **month index** (added in v2 to partially absorb the learning curve — later months = more experience).

In [ ]:
X = df_trim[COVARIATES].values
y = df_trim['morning'].values

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_sc, y)

df_trim = df_trim.copy()
df_trim['ps']       = lr.predict_proba(X_sc)[:, 1]
eps = 1e-6
df_trim['log_odds'] = np.log((df_trim['ps'] + eps) / (1 - df_trim['ps'] + eps))

print(f'PS model train accuracy: {lr.score(X_sc, y):.3f}')
print()
print('Coef magnitudes (standardised):')
for name, coef in zip(COVARIATES, lr.coef_[0]):
    print(f'  {name:20s}: {coef:+.4f}')
print()
print('Propensity score by group:')
print(df_trim.groupby('morning')['ps'].describe().round(4)
        .rename(index={0: 'Evening', 1: 'Morning'}))

In [ ]:
# ── 1:1 NN matching with caliper ───────────────────────────────────────────
caliper = 0.2 * df_trim['log_odds'].std()
print(f'Caliper: {caliper:.5f}')

treated = df_trim[df_trim['morning'] == 1].reset_index(drop=True)
control = df_trim[df_trim['morning'] == 0].reset_index(drop=True)

nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
nbrs.fit(control[['log_odds']].values)
distances, indices = nbrs.kneighbors(treated[['log_odds']].values)

mask = distances.flatten() <= caliper
treated_m = treated[mask].copy()
control_m = control.iloc[indices.flatten()[mask]].copy()

print(f'Matched pairs:     {mask.sum():,}')
print(f'Dropped (caliper): {(~mask).sum():,} morning reviews')
print()
print('Matched sample response time:')
for name, g in [('Morning', treated_m), ('Evening', control_m)]:
    print(f'  {name}: median={g["time"].median():.0f}ms  mean={g["time"].mean():.0f}ms  n={len(g)}')

In [ ]:
# ── Covariate balance ──────────────────────────────────────────────────────
def smd(a, b):
    return abs(a.mean() - b.mean()) / (np.sqrt((a.std()**2 + b.std()**2) / 2) + 1e-9)

balance = []
for cov in COVARIATES:
    balance.append({
        'covariate': cov,
        'SMD_before': smd(treated[cov], control[cov]),
        'SMD_after':  smd(treated_m[cov], control_m[cov]),
    })
bal_df = pd.DataFrame(balance)
print('Covariate Balance (SMD, target < 0.1):')
print(bal_df.to_string(index=False))

---
## Figure 2: Balance & Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── Balance plot ───────────────────────────────────────────────────────────
x = np.arange(len(COVARIATES))
axes[0].bar(x - 0.2, bal_df['SMD_before'], 0.35,
            label='Before', color=EVENING_COLOR, alpha=0.8)
axes[0].bar(x + 0.2, bal_df['SMD_after'],  0.35,
            label='After',  color=MORNING_COLOR, alpha=0.8)
axes[0].axhline(0.1, ls='--', color='black', lw=1, label='Threshold (0.1)')
axes[0].set_xticks(x)
axes[0].set_xticklabels([c.replace('_', '\n') for c in COVARIATES], fontsize=8)
axes[0].set_ylabel('Absolute SMD')
axes[0].set_title('Covariate Balance\nBefore vs After Matching')
axes[0].legend(frameon=False, fontsize=8)

# ── Pre-match distributions ────────────────────────────────────────────────
for grp, label, color in [(1, 'Morning', MORNING_COLOR), (0, 'Evening', EVENING_COLOR)]:
    vals = df_trim[df_trim['morning'] == grp]['time'] / 1000
    axes[1].hist(vals, bins=60, alpha=0.45, color=color, label=label,
                 density=True, range=(0, 30))
    axes[1].axvline(vals.median(), color=color, lw=2, ls='--')
axes[1].set_xlabel('Response time (seconds)')
axes[1].set_ylabel('Density')
axes[1].set_title('Before Matching\n(dashed = median, clipped at 30s)')
axes[1].legend(frameon=False)
axes[1].set_xlim(0, 30)

# ── Post-match distributions ───────────────────────────────────────────────
for times, label, color in [
    (treated_m['time'].values, 'Morning', MORNING_COLOR),
    (control_m['time'].values, 'Evening', EVENING_COLOR)
]:
    axes[2].hist(times / 1000, bins=60, alpha=0.45, color=color, label=label,
                 density=True, range=(0, 30))
    axes[2].axvline(np.median(times) / 1000, color=color, lw=2, ls='--')
axes[2].set_xlabel('Response time (seconds)')
axes[2].set_ylabel('Density')
axes[2].set_title('After Matching\n(dashed = median, clipped at 30s)')
axes[2].legend(frameon=False)
axes[2].set_xlim(0, 30)

plt.suptitle('Hard Reviews (ef < 2.5 at Review Time) — v2 PSM', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('v2_fig2_distributions.png', bbox_inches='tight')
plt.show()

---
## ATE Estimation

In [ ]:
morning_t = treated_m['time'].values
evening_t = control_m['time'].values

ate_mean   = morning_t.mean()        - evening_t.mean()
ate_median = np.median(morning_t)    - np.median(evening_t)

rng = np.random.default_rng(42)
n   = len(morning_t)
boot_mean, boot_med = [], []
for _ in range(10_000):
    idx = rng.integers(0, n, n)
    boot_mean.append(morning_t[idx].mean()      - evening_t[idx].mean())
    boot_med.append(np.median(morning_t[idx])   - np.median(evening_t[idx]))

ci_mean_lo, ci_mean_hi = np.percentile(boot_mean, [2.5, 97.5])
ci_med_lo,  ci_med_hi  = np.percentile(boot_med,  [2.5, 97.5])

t_stat, p_ttest = stats.ttest_ind(morning_t, evening_t)
u_stat, p_mw    = stats.mannwhitneyu(morning_t, evening_t, alternative='two-sided')

print('=' * 62)
print('ATE — Morning vs Evening Response Time (Hard Reviews, v2)')
print('=' * 62)
print(f'  Definition:    revlog.factor < 2.5 at time of review')
print(f'  Matched pairs: Morning n={len(morning_t):,}  |  Evening n={len(evening_t):,}')
print()
print(f'  Morning:  mean={morning_t.mean():.0f}ms  median={np.median(morning_t):.0f}ms')
print(f'  Evening:  mean={evening_t.mean():.0f}ms  median={np.median(evening_t):.0f}ms')
print()
print(f'  ATE (mean diff):   {ate_mean:+.0f}ms  95% CI [{ci_mean_lo:+.0f}, {ci_mean_hi:+.0f}]')
print(f'  ATE (median diff): {ate_median:+.0f}ms  95% CI [{ci_med_lo:+.0f}, {ci_med_hi:+.0f}]')
print()
print(f'  t-test:          t={t_stat:.3f},  p={p_ttest:.4f}')
print(f'  Mann-Whitney U:  U={u_stat:.0f},  p={p_mw:.4f}')
print()
if p_mw < 0.05:
    direction = 'SLOWER' if ate_median > 0 else 'FASTER'
    print(f'  --> Morning reviews significantly {direction} on hard cards (p={p_mw:.4f})')
else:
    print(f'  --> No significant difference in response time (p={p_mw:.4f})')

print()
print('--- Comparison with v1 ---')
print('  v1 (cards.factor ≤ 2.65, current snapshot):')
print('    ATE median = +1111ms, p < 0.0001')
print('  v2 (revlog.factor < 2.5, at review time):')
print(f'    ATE median = {ate_median:+.0f}ms, p = {p_mw:.4f}')

In [ ]:
# ── Figure 3: ATE forest plot comparing v1 and v2 ─────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

estimates = [
    ('v2 — Mean diff\n(at-review-time def.)',   ate_mean,   ci_mean_lo,  ci_mean_hi,  p_ttest),
    ('v2 — Median diff\n(at-review-time def.)',  ate_median, ci_med_lo,   ci_med_hi,   p_mw),
    ('v1 — Mean diff\n(current-state def.)',     1168,       225,         2108,        0.016),
    ('v1 — Median diff\n(current-state def.)',   1111,       557,         1634,        0.0001),
]

y_pos   = list(range(len(estimates)))
colors  = [MORNING_COLOR if p < 0.05 else 'grey' for *_, p in estimates]
markers = ['o', 'o', 's', 's']

for yp, (label, est, lo, hi, p), color, marker in zip(y_pos, estimates, colors, markers):
    ax.scatter(est, yp, s=90, color=color, zorder=3, marker=marker)
    ax.plot([lo, hi], [yp, yp], color=color, lw=2.5)
    ax.text(max(hi, 0) + 30, yp,
            f'{est:+.0f}ms  p={p:.4f}',
            va='center', fontsize=8.5, color=color)

ax.axvline(0, ls='--', color='black', lw=1)
ax.set_yticks(y_pos)
ax.set_yticklabels([e[0] for e in estimates], fontsize=9)
ax.set_xlabel('ATE in milliseconds  [Morning - Evening]\nPositive = morning is slower')
ax.set_title('v1 vs v2: Effect of Hard-Card Definition on ATE')

# Shaded zero line region
ax.axvspan(-200, 200, alpha=0.06, color='grey')
ax.text(0, -0.7, 'no difference', ha='center', fontsize=7, color='grey')

all_vals = [lo for *_, lo, hi, _ in estimates] + [hi for *_, lo, hi, _ in estimates]
ax.set_xlim(min(all_vals) - 300, max(all_vals) + 900)

plt.tight_layout()
plt.savefig('v2_fig3_comparison.png', bbox_inches='tight')
plt.show()

---
## Summary

### What changed with the corrected definition

| | v1 (current-state) | v2 (at-review-time) |
|---|---|---|
| Hard-review definition | cards.factor ≤ 2.65 today | revlog.factor < 2.5 at moment of review |
| Matched pairs | 533 | reported above |
| ATE median | +1,111 ms | reported above |
| Mann-Whitney p | < 0.0001 | reported above |

### Residual limitations
1. **Learning curve not fully absorbed.** `month_idx` is included as a confounder but is a rough proxy. A more rigorous approach would restrict to a narrower time window (e.g., only 2026) where skill level is stable.
2. **Zero morning reviews Jan–Apr 2025.** You did not review in the morning during your first four months. All morning hard-card reviews start from May 2025. This means the matched morning and evening reviews are drawn from different calendar periods even after PSM.
3. **Right-skewed response times.** Mann-Whitney is the more reliable test; t-test and mean-based ATE should be treated as secondary.